# 市场广度 / TRIN 曲线与检验

读取 `data/` 下的两个 CSV，输出图片到 `charts/`，并重写 `README.md`（仓库首页直接看结果）。

TRIN 用标准 Arms Index：(上涨家数 / 下跌家数) / (上涨成交量 / 下跌成交量)。
参考线：Arms 原始 10 日均线阈值 0.8（超买）/ 1.2（超卖）；博主的单日阈值 0.7（他称"多头主动进攻"）。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager

for f in ["Noto Sans CJK SC", "Noto Sans CJK JP", "WenQuanYi Zen Hei", "SimHei",
          "Microsoft YaHei", "PingFang SC", "Heiti SC", "Arial Unicode MS"]:
    if any(x.name == f for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.sans-serif"] = [f]
        break
plt.rcParams.update({
    "axes.unicode_minus": False, "figure.dpi": 110, "savefig.dpi": 150,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": "#0b0b0b",
    "legend.frameon": False, "legend.fontsize": 9,
})
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
INK2, MUTED = "#52514e", "#898781"

os.makedirs("charts", exist_ok=True)
b = pd.read_csv("data/a_breadth.csv", dtype={"trade_date": str})
b["date"] = pd.to_datetime(b.trade_date)
b = b.sort_values("date").reset_index(drop=True)
fut = pd.read_csv("data/cffex_holding.csv", dtype={"trade_date": str}) if os.path.exists("data/cffex_holding.csv") else pd.DataFrame()

KEY = {"20260323": "3/23", "20260721": "7/21", "20260921": "9/21",
       "20260922": "9/22", "20261022": "10/22", "20261023": "10/23"}
key = b[b.trade_date.isin(KEY)].copy()
key["label"] = key.trade_date.map(KEY)
last = b.iloc[-1]
print(f"数据区间 {b.trade_date.iloc[0]} ~ {last.trade_date}，共 {len(b)} 个交易日")


def mark_keys(ax, top=False):
    prev = None
    for i, (_, r) in enumerate(key.iterrows()):
        ax.axvline(r.date, color=MUTED, lw=0.8, ls=":", zorder=0)
        if top:
            near = prev is not None and (r.date - prev).days < 20
            dy = -14 if near and i % 2 else -2
            ax.annotate(r.label, (r.date, 1), xycoords=("data", "axes fraction"),
                        xytext=(2, dy), textcoords="offset points", fontsize=8, color=INK2, va="top")
            prev = r.date


def fmt_x(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))


def trin_axis(ax):
    ax.set_yscale("log")
    ax.set_yticks([0.3, 0.5, 0.7, 0.8, 1, 1.2, 1.5, 2, 3])
    ax.get_yaxis().set_major_formatter(matplotlib.ticker.FormatStrFormatter("%g"))
    ax.get_yaxis().set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.axhline(1, color=MUTED, lw=0.8)
    ax.axhline(0.8, color=GREEN, lw=0.8, ls="--")
    ax.axhline(1.2, color=RED, lw=0.8, ls="--")


## 图 1：上证指数 / 市场广度 / TRIN（单日 + 10 日均线）

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, gridspec_kw={"height_ratios": [1.2, 1, 1.2]})
ax = axes[0]
ax.plot(b.date, b.sh_close, color=BLUE, lw=1.6)
ax.set_title("上证指数")
mark_keys(ax, top=True)
ax = axes[1]
ax.plot(b.date, b.breadth_up * 100, color=MUTED, lw=0.7)
ax.plot(b.date, (b.breadth_up * 100).rolling(10).mean(), color=AQUA, lw=2, label="10 日均线")
ax.set_title("市场广度：上涨家数占比（%）")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")
mark_keys(ax)
ax = axes[2]
ax.plot(b.date, b.trin, color=MUTED, lw=0.7, label="单日 TRIN")
ax.plot(b.date, b.trin_ma10, color=BLUE, lw=2, label="10 日均线")
trin_axis(ax)
ax.set_title("TRIN（对数坐标；虚线 0.8 超买 / 1.2 超卖，Arms 原始阈值）")
ax.legend(loc="upper left", ncol=2)
mark_keys(ax)
fmt_x(ax)
fig.tight_layout()
fig.savefig("charts/1_overview.png")
plt.show()


## 图 2：博主提到的几天——家数占比 vs 成交量占比 vs 成交额

如果"上涨股爆量"，上涨股的成交量占比应明显高于家数占比。TRIN 只反映这两个占比的相对关系，不反映绝对成交量。

In [ ]:
if not key.empty:
    k = key.copy()
    k["cnt_share"] = k.n_up / (k.n_up + k.n_dn) * 100
    k["vol_share"] = k.up_vol / (k.up_vol + k.dn_vol) * 100
    k["amt_share"] = k.up_amt / (k.up_amt + k.dn_amt) * 100
    x = np.arange(len(k))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.6, 1]})
    ax = axes[0]
    w = 0.26
    for i, (col, name, c) in enumerate([("cnt_share", "上涨家数占比", MUTED),
                                         ("vol_share", "上涨股成交量占比", BLUE),
                                         ("amt_share", "上涨股成交额占比", ORANGE)]):
        bars = ax.bar(x + (i - 1) * w, k[col], w * 0.92, color=c, label=name)
        ax.bar_label(bars, fmt="%.0f", fontsize=8, color=INK2, padding=2)
    ax.set_xticks(x, [f"{l}\nTRIN {t:.2f}" for l, t in zip(k.label, k.trin)])
    ax.set_ylim(0, 115)
    ax.set_title("上涨股：家数占比 vs 成交量 / 成交额占比（%）")
    ax.legend(loc="upper left", ncol=3)
    ax.grid(axis="x", visible=False)
    ax = axes[1]
    med = b.amt.rolling(60).median()
    bars = ax.bar(x, k.amt, 0.6, color=BLUE)
    ax.bar_label(bars, fmt="%.0f", fontsize=8, color=INK2, padding=2)
    ax.scatter(x, med[k.index], color=ORANGE, marker="_", s=600, lw=2, zorder=3, label="此前 60 日成交额中位数")
    ax.set_xticks(x, k.label)
    ax.set_ylim(0, max(k.amt.max(), med[k.index].max()) * 1.22)
    ax.set_title("当日两市成交额（亿元）")
    ax.legend(loc="upper left")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig.savefig("charts/2_key_days.png")
    plt.show()


## 图 3：权重（市值前 300）vs 中小盘的 TRIN 10 日均线

检验"权重放量出货、小票补涨"：如果成立，权重组 TRIN 应明显高于中小盘组。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.4]})
ax = axes[0]
ax.plot(b.date, b.sh_close, color=BLUE, lw=1.6)
ax.set_title("上证指数")
mark_keys(ax, top=True)
ax = axes[1]
ax.plot(b.date, b.big_trin_ma10, color=VIOLET, lw=2, label="权重（市值前 300）")
ax.plot(b.date, b.small_trin_ma10, color=ORANGE, lw=2, label="中小盘（其余）")
trin_axis(ax)
ax.set_title("分组 TRIN 10 日均线（对数坐标）")
ax.legend(loc="upper left", ncol=2)
mark_keys(ax)
fmt_x(ax)
fig.tight_layout()
fig.savefig("charts/3_big_vs_small.png")
plt.show()


## 图 4：中信期货（代客）净持仓 vs 前 20 会员合计

净持仓 = 多单 − 空单（手）。数据是中金所公布的会员排名，"代客"即客户持仓合计。

In [ ]:
if not fut.empty:
    fut["net"] = fut.long_hld - fut.short_hld
    fut["date"] = pd.to_datetime(fut.trade_date)
    citic = fut[fut.broker.astype(str).str.contains("中信期货")].groupby(["date", "product"]).net.sum().unstack()
    total = fut.groupby(["date", "product"]).net.sum().unstack()
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    ax = axes[0]
    ax.plot(b.date, b.hs300_close, color=BLUE, lw=1.6)
    ax.set_title("沪深300")
    colors = {"IF": BLUE, "IH": ORANGE, "IC": AQUA, "IM": VIOLET}
    for ax, df, title in [(axes[1], citic, "中信期货（代客）净持仓（手）"), (axes[2], total, "前 20 会员净持仓合计（手）")]:
        for p in ["IF", "IH", "IC", "IM"]:
            if p in df:
                ax.plot(df.index, df[p], color=colors[p], lw=1.6, label=p)
        ax.axhline(0, color=MUTED, lw=0.8)
        ax.set_title(title)
        ax.legend(loc="lower left", ncol=4)
    fmt_x(axes[2])
    fig.tight_layout()
    fig.savefig("charts/4_cffex.png")
    plt.show()


## 图 5：规则检验——按 TRIN 区间分组，之后上证指数的平均涨跌幅

博主规则：单日 TRIN < 0.7 偏多、0.9 左右中性。Arms 原始规则：10 日均线 < 0.8 超买、> 1.2 超卖。
这里只看历史统计，样本量小、区间重叠，结论仅供参考。

In [ ]:
for h in (5, 20):
    b[f"fwd{h}"] = (b.sh_close.shift(-h) / b.sh_close - 1) * 100
bins_d = [0, 0.5, 0.7, 1.0, 1.2, 2.0, np.inf]
lab_d = ["<0.5", "0.5–0.7", "0.7–1.0", "1.0–1.2", "1.2–2.0", ">2.0"]
bins_m = [0, 0.8, 1.0, 1.2, np.inf]
lab_m = ["<0.8", "0.8–1.0", "1.0–1.2", ">1.2"]
rows = []
for col, bins, labs, name in [("trin", bins_d, lab_d, "单日 TRIN"), ("trin_ma10", bins_m, lab_m, "TRIN 10 日均线")]:
    g = b.assign(bucket=pd.cut(b[col], bins, labels=labs)).groupby("bucket", observed=False)
    for lab, d in g:
        rows.append({"口径": name, "区间": lab, "天数": int(d.fwd20.notna().sum()),
                     "5日后平均涨跌%": d.fwd5.mean(), "20日后平均涨跌%": d.fwd20.mean(),
                     "20日后上涨概率%": (d.fwd20 > 0).mean() * 100 if d.fwd20.notna().any() else np.nan})
bt = pd.DataFrame(rows)
base = {"5日后平均涨跌%": b.fwd5.mean(), "20日后平均涨跌%": b.fwd20.mean(), "20日后上涨概率%": (b.fwd20 > 0)[b.fwd20.notna()].mean() * 100}
bt.round(2).to_csv("data/trin_backtest.csv", index=False, encoding="utf-8-sig")
print(bt.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
for ax, name in zip(axes, ["单日 TRIN", "TRIN 10 日均线"]):
    d = bt[bt["口径"] == name].reset_index(drop=True)
    x = np.arange(len(d))
    for i, (col, c) in enumerate([("5日后平均涨跌%", MUTED), ("20日后平均涨跌%", BLUE)]):
        bars = ax.bar(x + (i - 0.5) * 0.38, d[col], 0.36, color=c, label=col.replace("平均涨跌%", "平均涨跌"))
        ax.bar_label(bars, fmt="%.1f", fontsize=8, color=INK2, padding=2)
    ax.axhline(base["20日后平均涨跌%"], color=ORANGE, lw=1.2, ls="--", label="全样本 20 日平均")
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_xticks(x, [f"{r}\n{n} 天" for r, n in zip(d["区间"], d["天数"])])
    ax.set_title(f"按 {name} 分组：之后上证平均涨跌（%）")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8)
    ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig("charts/5_backtest.png")
plt.show()


## 写 README（仓库首页显示）

In [ ]:
def pct(v):
    return "" if pd.isna(v) else f"{v * 100:.1f}%"


def num(v, f="{:.2f}"):
    return "" if pd.isna(v) else f.format(v)


def md_table(df):
    head = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "".join("| " + " | ".join(str(v) for v in r) + " |\n" for r in df.values)


kt = pd.DataFrame({
    "日期": key.label, "上涨家数占比": key.breadth_up.map(pct), "涨停": key.get("limit_up", pd.Series(dtype=float)).map(lambda v: num(v, "{:.0f}")),
    "TRIN": key.trin.map(num), "TRIN(成交额)": key.trin_amt.map(num), "TRIN 10日均": key.trin_ma10.map(num),
    "权重 TRIN": key.big_trin.map(num), "中小盘 TRIN": key.small_trin.map(num), "成交额(亿)": key.amt.map(lambda v: num(v, "{:.0f}")),
})
recent = b.tail(10)
rt = pd.DataFrame({
    "日期": recent.trade_date, "上证": recent.sh_close.map(lambda v: num(v)), "上涨家数占比": recent.breadth_up.map(pct),
    "TRIN": recent.trin.map(num), "TRIN 10日均": recent.trin_ma10.map(num),
    "权重 TRIN": recent.big_trin.map(num), "中小盘 TRIN": recent.small_trin.map(num),
    "成交额(亿)": recent.amt.map(lambda v: num(v, "{:.0f}")),
})
readme = f"""# A 股市场广度 / TRIN 跟踪

自动更新：每个交易日收盘后由 GitHub Actions 运行。最新数据日期：**{last.trade_date}**（共 {len(b)} 个交易日，起始 {b.trade_date.iloc[0]}）。

**TRIN（Arms Index）=（上涨家数 ÷ 下跌家数）÷（上涨成交量 ÷ 下跌成交量）**，平盘不计。小于 1 表示成交量相对集中在上涨股。
Arms 原始用法看 10 日均线：低于 0.8 视为超买，高于 1.2 视为超卖。范围：沪深 A 股（不含北交所）。

## 最近 10 个交易日

{md_table(rt.iloc[::-1])}
## 博主提到的几天

{md_table(kt) if not kt.empty else "数据区间内暂无这些日期。"}
## 图表

![总览](charts/1_overview.png)

![关键日期](charts/2_key_days.png)

![权重 vs 中小盘](charts/3_big_vs_small.png)

![中金所持仓](charts/4_cffex.png)

![规则检验](charts/5_backtest.png)

规则检验明细见 [`data/trin_backtest.csv`](data/trin_backtest.csv)。

## 使用

1. 仓库 **Settings → Secrets and variables → Actions → New repository secret**，名称 `TUSHARE_TOKEN`，值填 tushare token。
2. **Actions → 更新 TRIN 数据 → Run workflow** 手动运行一次（首次从 2024 年拉起，约 30–60 分钟）。之后每个交易日北京时间 18:00 左右自动运行，只拉新增日期。
3. 运行完成后刷新本页即可看到最新结果；执行过的 notebook 也会提交回仓库。

文件：`pull_data.ipynb` 拉数据 → `data/*.csv`；`charts.ipynb` 出图 → `charts/*.png` 并重写本 README。
"""
open("README.md", "w", encoding="utf-8").write(readme)
print("README.md 已更新")
